In [ ]:
import pandas as pd
import numpy as np
import time
import pickle 

from sbi.utils import BoxUniform
from scipy.integrate import odeint

import torch
from torch.distributions import Uniform, Exponential, Cauchy

_ = torch.manual_seed(0)
_ = np.random.seed(0) 

device = "cuda:0" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

/etc/python/sitecustomize.py:236: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


In [3]:
def poisson_noise(simulation):
    
    with_noise = np.random.poisson(np.maximum(simulation, 1e-6))

    return with_noise

In [ ]:
with open('../../Data/Model3/M3_dataset.pkl', 'rb') as handle:
    sim_traj = pickle.load(handle)

with open('../../Data/Model3/M3_params.pkl', 'rb') as handle:
    sim_params = pickle.load(handle)
    
with open('../../Data/Model3/sim_dataset.pkl', 'rb') as f:
    simulation_dataset = pickle.load(f)

In [ ]:
param_names = ["beta", "phi_s", "g_rate"]

In [ ]:
traj   = torch.tensor(poisson_noise(sim_traj.numpy()), dtype=torch.float32)
params = sim_params

In [ ]:
import torch.nn as nn
from sbi.inference import NPE
from sbi.neural_nets import posterior_nn

In [16]:
low  = torch.tensor([0.6, 0.2, 0.005])
high = torch.tensor([1.2, 0.6, 0.01])

prior = BoxUniform(low=low, high=high)

In [ ]:
xs     = traj
thetas = params

mean = xs.mean(axis=(0,1))
std = xs.std(axis=(0,1))+1e-6

xs_norm = (xs-mean)/std
xs_norm = torch.tensor(xs_norm, dtype=torch.float32)

xs_norm = xs_norm.reshape(len(thetas),-1)

/tmp/ipykernel_273814/1354707161.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  xs_norm = torch.tensor(xs_norm, dtype=torch.float32)


In [ ]:
neural_posterior = posterior_nn(model='maf')

inference = NPE(density_estimator=neural_posterior,device=device)

# training
density_estimator=inference.append_simulations(thetas, xs_norm).train(training_batch_size=256)

posterior = inference.build_posterior(density_estimator)

In [ ]:
posterior_samples=[]

for i, sim in enumerate(simulation_dataset):  
    x_obs = torch.as_tensor(sim['poisson'], dtype=torch.float32).to(device)
    x_obs_norm = (x_obs-mean)/std
    x_obs_norm = x_obs_norm.reshape(-1)
    
    samples = posterior.sample((10000,), x=x_obs_norm)
    samples_df=pd.DataFrame(samples.cpu().numpy(),columns=param_names)
    posterior_samples.append(samples_df)